Use this cell for evaluating the results of experiments

In [35]:
import pandas as pd

def make_markdown(df):
    lines = [[f"{col}" for col in df.columns], ["-" for col in df.columns]]
    lines += [[f"{value}" for value in df.iloc[i]] for i in range(df.shape[0])]
    lines = ["|" + "|".join(line) + "|" for line in lines]
    final = "\n".join(lines)
    return final

def keep_top_n_rows(df, n=15, score_cols=None, index_col="features"):
    """Keep rows in the top `n` of at least one score column. The `features`
    (string) column is moved to the index so it's preserved, then restored."""
    out = df.set_index(index_col) if index_col and index_col in df.columns else df

    if score_cols is None:
        num = out.select_dtypes("number")          # ranks on numeric cols only
        score_cols = [c for c in num.columns if not c.endswith("_pct")]

    keep = set()
    for c in score_cols:
        keep |= set(out[c].nlargest(n).index)

    out = out.loc[out.index.isin(keep)]
    return out.reset_index()                        # 'features' back as a column

n = 12
def print_tables(exp):
    filename = f"experiments/test_results/_experiment{exp}"
    res = pd.read_csv(filename + ".csv")
    imp1 = pd.read_csv(filename + "_importance.csv")
    imp2 = pd.read_csv(filename + "_importance_perm.csv")
    top1 = keep_top_n_rows(imp1, n=n)
    top2 = keep_top_n_rows(imp2, n=n)
    print("\n#### Results")
    print(make_markdown(res))
    print(f"\n#### Feature Importance XGBoost\n(Top {n} kept for each column)")
    print(make_markdown(top1))
    print(f"\n#### Feature Importance Col Shuffle\n(Top {n} kept for each column)")
    print(make_markdown(top2))

In [40]:
print_tables(exp="7")


#### Results
|recipe|n_sites|median_r2|mean_r2|
|-|-|-|-|
|A_static|20|-0.0248873320155595|-0.641397738975395|
|B_static|20|0.8779130870020047|0.7513493106700608|
|C_static|20|0.5618503174004563|0.4247365245935888|
|D_static|20|0.4546353715630746|0.2854733279150639|

#### Feature Importance XGBoost
(Top 12 kept for each column)
|features|A_static|A_static_pct|B_static|B_static_pct|C_static|C_static_pct|D_static|D_static_pct|
|-|-|-|-|-|-|-|-|-|
|USGS-05482300_lag2|nan|nan|0.28880247|2.0698202|nan|nan|nan|nan|
|USGS-05482500_lag2|nan|nan|0.28468436|2.040306|nan|nan|nan|nan|
|USGS-05464420_lag2|nan|nan|0.28327346|2.0301943|nan|nan|nan|nan|
|USGS-05484500_lag2|nan|nan|0.27970624|2.0046284|nan|nan|nan|nan|
|USGS-05484500_lag1|nan|nan|0.40633783|2.912185|0.016541358|1.6290021|nan|nan|
|USGS-05412500_lag1|nan|nan|0.35250717|2.5263858|0.00757526|0.7460158|nan|nan|
|USGS-05482500_lag1|nan|nan|0.3345037|2.3973567|0.010669898|1.0507774|nan|nan|
|WQS0024_lag1|nan|nan|0.331601|2.3765533|0.0029850

[1, 2, 3, 4, 5]